# MedGraphRAG-style Knowledge Graph — Harper's Biochemistry (all rewrite chapters)

Builds a knowledge graph from every rewritten chapter in `data/textbook/rewrite/`
(Harper's Illustrated Biochemistry, 33rd Edition — LLM-rewritten, tagged
`<sec>/<con>/<fig>/<tbl>/...` markdown), following the graph-construction method
from **MedGraphRAG** ([arXiv:2408.04187](https://arxiv.org/abs/2408.04187)):
semantic chunking, entity extraction, a tri-tier linked graph, relation
extraction, tag-based hierarchical clustering, and a minimal U-Retrieval query
layer.

**Why not `src/domain_kg/`?** That pipeline's Stage 1 parser (`stage1_parse.py`)
does expect exactly this tagged format, and would be the more "correct"
long-term tool — but its NER/UMLS-linking stages (Stage 2/3) need scispaCy +
faiss, which only ship wheels for Python 3.9–3.11 and have no committed setup
recipe in this repo (see `docs/08-domain-kg-pipeline.md`). This notebook
instead implements MedGraphRAG's steps directly against the tagged chapter
text using a **local, open-source LLM** — Qwen3.5-8B, run via `transformers`
on-device (same model + loading pattern as this repo's `mcq_generation.ipynb`
`TripleExtractor`: lazy load/unload, `enable_thinking=False`, JSON-repair
parsing). No API key, no external network calls. 8B (not the larger 27B) was
chosen because every task here is structured JSON extraction rather than
open-ended reasoning, and the corpus is 658 chunks across 32 chapters — 8B's
schema-following is sufficient and the speed/memory win matters more at this
scale than a marginal quality gain would.

Considered and rejected for this notebook: `OpenMeditron/Meditron3-8B` (base
checkpoint, not instruction-tuned — needs schema-constrained decoding just to
stay on-task, per `domain_kg`'s own docstring) and
`mlx-community/HuatuoGPT-o1-72B-4bit` (medical-specialist reasoning model —
better ceiling on open-ended clinical reasoning, but slower per call from
reasoning traces on every extraction step, which this notebook's tasks are
mostly too simple to need — also the same model already used to *generate*
the rewrite corpus, per each file's `<src>` meta line, so re-using it here
would double a single model's biases rather than cross-check them). Qwen3.5
is the generalist already proven in this repo for this exact kind of
physiology/biochemistry-textbook extraction task.

**Tri-tier mapping used here** (paper section 2.1.3):
- **Tier 1** — entities extracted from every rewrite chapter (the "user RAG data").
- **Tier 2** — source reference. The chapters *are* the authoritative textbook
  already, so Tier 2 is each entity's originating chunk/section in Harper's —
  no separate corpus needed, linked deterministically via `the_reference_of`.
- **Tier 3** — dictionary definitions. No UMLS/MRCONSO license in this repo
  (same caveat `domain_kg`'s docs already carry), so this tier is an
  LLM-generated canonical definition per unique entity name, standing in
  for a real UMLS lookup, linked via `the_definition_of`. **Not clinically
  authoritative — do not treat as verified UMLS/SNOMED linkage.**

This is a prototype notebook, not wired into `app/` or `src/domain_kg/`.

## Setup

In [ ]:
import gc
import json
import logging
import os
import re
from dataclasses import dataclass, field
from pathlib import Path

import torch
from transformers import AutoModelForMultimodalLM, AutoProcessor

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

REWRITE_DIR = REPO_ROOT / "data" / "textbook" / "rewrite"
CHAPTER_PATHS = sorted(REWRITE_DIR.glob("chapter_*.md"))
assert CHAPTER_PATHS, f"no chapter files found under {REWRITE_DIR}"

OUTPUT_DIR = REPO_ROOT / "notebooks" / "output" / "medgraphrag_kg"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

log = logging.getLogger("medgraphrag_kg")


def get_device() -> str:
    if torch.cuda.is_available():
        return "cuda"
    if torch.backends.mps.is_available():
        return "mps"
    return "cpu"


_THINK_BLOCK_RE = re.compile(r"<think>.*?</think>", re.DOTALL)
_JSON_BLOCK_RE = re.compile(r"\{.*\}", re.DOTALL)
_FENCE_RE = re.compile(r"^```(json)?|```$", re.MULTILINE)


def _repair_truncated_json(raw: str) -> str:
    """Best-effort fix for JSON cut off mid-generation (max_new_tokens truncation): drops
    back to the last fully-formed value (closing an unterminated string first if needed),
    then closes whatever braces/brackets are still open — in the correct nesting order —
    so json.loads has a chance at the still-complete content. Ported from
    mcq_generation.ipynb's TripleExtractor."""
    text = raw[raw.index("{"):] if "{" in raw else raw

    if text.count('"') % 2 == 1:
        text = text[: text.rindex('"')]

    last_safe = max(text.rfind("}"), text.rfind("]"), text.rfind(","))
    if last_safe == -1:
        raise ValueError("no safe truncation point found")
    text = text[:last_safe] if text[last_safe] == "," else text[: last_safe + 1]

    stack: list[str] = []
    in_string = False
    escape = False
    for ch in text:
        if in_string:
            if escape:
                escape = False
            elif ch == "\\":
                escape = True
            elif ch == '"':
                in_string = False
            continue
        if ch == '"':
            in_string = True
        elif ch in "{[":
            stack.append(ch)
        elif ch in "}]":
            stack.pop()

    closers = {"{": "}", "[": "]"}
    return text + "".join(closers[ch] for ch in reversed(stack))


class LocalLLM:
    """Qwen3.5-8B, loaded lazily and released after use — same pattern as
    mcq_generation.ipynb's `TripleExtractor` / `src/captioning/qwen_vl.py`, one model
    resident at a time on the unified-memory budget. Qwen3.5 is a multimodal checkpoint
    (AutoModelForMultimodalLM + AutoProcessor rather than AutoModelForCausalLM +
    AutoTokenizer) but runs fine text-only. 8B chosen over 27B for this notebook's
    full-corpus run (658 chunks across all `data/textbook/rewrite/` chapters): every task
    here is structured JSON extraction, not open-ended reasoning, so 8B's schema-following
    is sufficient and the speed/memory win matters a lot more at this chunk count.

    `enable_thinking=False`: every call in this notebook is structured extraction, not
    open-ended reasoning, so thinking mode would just burn tokens and complicate parsing.
    `<think>` stripping is kept anyway as a defensive fallback in case the template ignores
    the flag.
    """

    MODEL_PATH = "Qwen/Qwen3.5-8B"

    def __init__(self, model_path: str = MODEL_PATH):
        self.model_path = model_path
        self.device = get_device()
        self.model = None
        self.processor = None

    def load(self):
        if self.model is not None:
            return
        self.processor = AutoProcessor.from_pretrained(self.model_path)
        self.model = AutoModelForMultimodalLM.from_pretrained(self.model_path, dtype=torch.bfloat16)
        self.model.to(self.device)
        self.model.eval()

    def unload(self):
        self.model = None
        self.processor = None
        gc.collect()
        if torch.backends.mps.is_available():
            torch.mps.empty_cache()
        elif torch.cuda.is_available():
            torch.cuda.empty_cache()

    def _generate(self, system: str, user: str, max_new_tokens: int) -> str:
        assert self.model is not None, "call load() first"
        messages = [{"role": "system", "content": system}, {"role": "user", "content": user}]
        inputs = self.processor.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors="pt",
            enable_thinking=False,
        ).to(self.device)
        with torch.no_grad():
            output_ids = self.model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                temperature=None,
                top_p=None,
                top_k=None,
            )
        new_tokens = output_ids[0][inputs["input_ids"].shape[1] :]
        return self.processor.decode(new_tokens, skip_special_tokens=True)

    def call_json(self, system: str, user: str, max_new_tokens: int = 2048, max_retries: int = 2) -> str:
        """Returns a raw JSON string, repaired if truncated. Callers pydantic-validate it."""
        last_err: Exception | None = None
        for attempt in range(max_retries + 1):
            raw = self._generate(system, user, max_new_tokens=max_new_tokens)
            cleaned = _FENCE_RE.sub("", _THINK_BLOCK_RE.sub("", raw)).strip()
            matches = _JSON_BLOCK_RE.findall(cleaned)
            if not matches:
                last_err = ValueError(f"no JSON object found in model output: {raw[:200]!r}")
                log.warning("call_json attempt %d/%d: %s", attempt + 1, max_retries + 1, last_err)
                continue
            candidate = matches[-1]
            try:
                json.loads(candidate)
                return candidate
            except json.JSONDecodeError:
                try:
                    repaired = _repair_truncated_json(candidate)
                    json.loads(repaired)
                    return repaired
                except (ValueError, json.JSONDecodeError) as e:
                    last_err = e
                    log.warning("call_json attempt %d/%d: %s", attempt + 1, max_retries + 1, e)
        assert last_err is not None
        raise last_err

    def call_text(self, system: str, user: str, max_new_tokens: int = 1024) -> str:
        raw = self._generate(system, user, max_new_tokens=max_new_tokens)
        return _THINK_BLOCK_RE.sub("", raw).strip()


llm = LocalLLM()
llm.load()

## Step 1 — Tag-based chunking

The paper's Step 1 uses an LLM sliding window to detect topic boundaries
paragraph-by-paragraph. `data/textbook/rewrite/` chapters already come
pre-segmented into `<sec id="..." level="1|2"><head>...</head><con>...</con>
...</sec>` blocks (LLM-rewritten by an external pipeline — see each file's
`<src>` meta line), so tag-based chunking gives the same outcome — one chunk
per topical section — far more cheaply and reliably than an LLM judging
boundaries again over already-clean structure. `<sec>` blocks in this corpus
are flat (never nested — verified: max stack depth 1 across the corpus), so
no recursive splitting is needed.

Each section's chunk text is the concatenation of its `<con>` (prose) blocks
in document order, with inline `<term>`/`<sup>`/`<sub>` tags stripped to their
text content and `<figref>`/`<tblref>`/`<cit>` stripped to their visible
label. `<fig>` (with nested `<cap>`/`<desc>`) and `<tbl>` blocks are pulled out
as `Passage`s (kept, not chunked as prose — mirrors `domain_kg`'s `Chunk` vs
`Passage` split) rather than fed to entity extraction. The chapter-level
`<sum>` and `<ref>` (bibliography) blocks, both outside any `<sec>`, are
dropped — same rationale as `mcq_generation.ipynb`'s reference/bibliography
exclusion.

In [ ]:
MAX_CHUNK_CHARS = 4000

SEC_RE = re.compile(r'<sec\s+id="([^"]*)"\s+level="(\d+)"\s*>(.*?)</sec>', re.DOTALL)
HEAD_RE = re.compile(r"<head>(.*?)</head>", re.DOTALL)
CON_RE = re.compile(r"<con>(.*?)</con>", re.DOTALL)
FIG_RE = re.compile(r"<fig\b[^>]*>.*?</fig>", re.DOTALL)
TBL_RE = re.compile(r"<tbl\b[^>]*>.*?</tbl>", re.DOTALL)
META_TITLE_RE = re.compile(r"<title>(.*?)</title>", re.DOTALL)
SRC_RE = re.compile(r"<src>(.*?)</src>", re.DOTALL)

# inline tags collapsed to their text content; refs/citations collapsed to their visible label
_INLINE_STRIP_RE = re.compile(r"</?(?:term|sup|sub)>")
_INLINE_LABEL_RE = re.compile(r"<(?:figref|tblref|cit)[^>]*>(.*?)</(?:figref|tblref|cit)>", re.DOTALL)
_TAG_RE = re.compile(r"<[^>]+>")


def _clean_prose(text: str) -> str:
    text = _INLINE_LABEL_RE.sub(r"\1", text)
    text = _INLINE_STRIP_RE.sub("", text)
    text = _TAG_RE.sub("", text)  # any remaining stray tags
    return re.sub(r"[ \t]+", " ", text).strip()


@dataclass
class Passage:
    """Non-assertable retained text (figures, tables) — kept with position, never chunked as prose."""

    doc_title: str
    chapter: str
    section: str
    html: str


@dataclass
class SourceChunk:
    uid: str
    doc_title: str
    chapter: str
    section: str
    text: str


def parse_chapter(path: Path) -> tuple[str, str, list[SourceChunk], list[Passage]]:
    raw = path.read_text(encoding="utf-8")

    title_m = META_TITLE_RE.search(raw)
    doc_title = "Harper's Illustrated Biochemistry, 33rd Edition"
    chapter = title_m.group(1).strip() if title_m else path.stem

    chunks: list[SourceChunk] = []
    passages: list[Passage] = []
    for idx, sec_m in enumerate(SEC_RE.finditer(raw)):
        body = sec_m.group(3)
        head_m = HEAD_RE.search(body)
        heading = _clean_prose(head_m.group(1)) if head_m else sec_m.group(1)

        for fig_m in FIG_RE.finditer(body):
            passages.append(Passage(doc_title=doc_title, chapter=chapter, section=heading, html=fig_m.group(0)))
        for tbl_m in TBL_RE.finditer(body):
            passages.append(Passage(doc_title=doc_title, chapter=chapter, section=heading, html=tbl_m.group(0)))

        # tables/figures already pulled above; only <con> blocks feed prose (skips fig/tbl's own nested <con>)
        body_wo_fig_tbl = TBL_RE.sub("", FIG_RE.sub("", body))
        prose = " ".join(_clean_prose(m.group(1)) for m in CON_RE.finditer(body_wo_fig_tbl))
        if not prose or len(prose) < 40:
            continue

        for j in range(0, len(prose), MAX_CHUNK_CHARS):
            part = prose[j : j + MAX_CHUNK_CHARS].strip()
            if not part:
                continue
            suffix = f"#{j // MAX_CHUNK_CHARS}" if j > 0 else ""
            chunks.append(
                SourceChunk(
                    uid=f"{path.stem}::{idx:03d}::{heading[:24]}{suffix}",
                    doc_title=doc_title,
                    chapter=chapter,
                    section=heading,
                    text=part,
                )
            )
    return doc_title, chapter, chunks, passages


all_chunks: list[SourceChunk] = []
all_passages: list[Passage] = []
for p in CHAPTER_PATHS:
    _, chapter, chunks, passages = parse_chapter(p)
    print(f"{chapter}: {len(chunks)} chunks, {len(passages)} figure/table passages")
    all_chunks.extend(chunks)
    all_passages.extend(passages)

print(f"\ntotal: {len(all_chunks)} chunks across {len(CHAPTER_PATHS)} chapters")

In [ ]:
# Sanity check — inspect a sample chunk
sample = all_chunks[len(all_chunks) // 2]
print(sample.uid)
print(f"chapter={sample.chapter!r} section={sample.section!r}\n")
print(sample.text[:500])

## Step 2 — Tier-1 entity extraction

Per chunk, one local Qwen3.5-8B call extracts entities as `{name, type, context}`
(paper 2.1.2), with `type` constrained to a UMLS-semantic-type-like taxonomy
rather than free text, so downstream tagging/clustering has a closed
vocabulary.

In [ ]:
from typing import Literal

from pydantic import BaseModel, Field, ValidationError

SEMANTIC_TYPES = (
    "Anatomical Structure",
    "Physiologic Function",
    "Cell or Cell Component",
    "Disease or Syndrome",
    "Sign or Symptom",
    "Pharmacologic Substance",
    "Diagnostic Procedure",
    "Therapeutic Procedure",
    "Biologically Active Substance",
    "Neoplastic Process",
    "Injury or Poisoning",
)
SemanticType = Literal[
    "Anatomical Structure",
    "Physiologic Function",
    "Cell or Cell Component",
    "Disease or Syndrome",
    "Sign or Symptom",
    "Pharmacologic Substance",
    "Diagnostic Procedure",
    "Therapeutic Procedure",
    "Biologically Active Substance",
    "Neoplastic Process",
    "Injury or Poisoning",
]


class Entity(BaseModel):
    name: str = Field(min_length=1)
    type: SemanticType
    context: str = Field(min_length=1, description="1-2 sentence context from the chunk")


class EntityExtractionResult(BaseModel):
    entities: list[Entity]


_ENTITY_SYSTEM_PROMPT = f"""You are a medical knowledge graph entity extractor. Read the given \
passage from a medical physiology textbook and extract all clinically/physiologically relevant \
named entities (anatomical structures, physiologic functions, cells, diseases, symptoms, drugs, \
procedures, etc).

Each entity's `type` must be exactly one of: {", ".join(SEMANTIC_TYPES)}.

`name` should be the entity's canonical/most specific name as it appears in or is implied by the \
text. `context` is a short 1-2 sentence excerpt or paraphrase grounding what role this entity \
plays in the passage.

Do not invent entities not supported by the text. Do not extract generic terms with no specific \
medical meaning (e.g. "body", "process", "function" on their own).

Respond with ONLY a JSON object: {{"entities": [{{"name": ..., "type": ..., "context": ...}}, ...]}}
No prose, no markdown fences, no explanation — JSON only."""


def extract_entities(chunk_text: str, max_new_tokens: int = 2048, max_retries: int = 2) -> EntityExtractionResult:
    last_err = None
    for _ in range(max_retries + 1):
        raw = llm.call_json(_ENTITY_SYSTEM_PROMPT, chunk_text, max_new_tokens=max_new_tokens)
        try:
            return EntityExtractionResult.model_validate_json(raw)
        except ValidationError as e:
            last_err = e
    raise last_err

In [ ]:
from tqdm.auto import tqdm


@dataclass
class ExtractedChunk:
    chunk: SourceChunk
    entities: list[Entity]


extracted_chunks: list[ExtractedChunk] = []
failures = 0
for chunk in tqdm(all_chunks, desc="extracting entities"):
    try:
        result = extract_entities(chunk.text)
    except Exception as e:
        failures += 1
        tqdm.write(f"skipping {chunk.uid}: {e}")
        continue
    extracted_chunks.append(ExtractedChunk(chunk=chunk, entities=result.entities))

print(f"extracted entities for {len(extracted_chunks)}/{len(all_chunks)} chunk(s), {failures} failure(s)")
print(f"total entity mentions: {sum(len(ec.entities) for ec in extracted_chunks)}")

In [ ]:
# Sanity check
for ec in extracted_chunks[:2]:
    print(ec.chunk.uid)
    for e in ec.entities[:5]:
        print(f"  - {e.name} ({e.type})")
    print()

## Step 3 — Tier-2 (source) and Tier-3 (dictionary) linking

**Tier 2** is deterministic: each Tier-1 entity's source reference is its
originating chunk's `(doc_title, chapter, section)` — no LLM/embedding needed
since the chapters already *are* the authoritative textbook.

**Tier 3** stands in for the paper's UMLS dictionary layer, which needs a
licensed MRCONSO release this repo doesn't have. Entity names are deduped
across all 3 chapters first so each unique name gets exactly one local-LLM
definition call, regardless of how many chunks mention it.

In [ ]:
@dataclass
class SourceRef:
    doc_title: str
    chapter: str
    section: str


@dataclass
class Tier1Entity:
    entity_id: str
    name: str
    type: str
    context: str
    chunk_uid: str
    source: SourceRef


def entity_key(name: str) -> str:
    return re.sub(r"\s+", " ", name.strip().lower())


tier1_entities: list[Tier1Entity] = []
for ec in extracted_chunks:
    for i, e in enumerate(ec.entities):
        tier1_entities.append(
            Tier1Entity(
                entity_id=f"{ec.chunk.uid}::e{i:03d}",
                name=e.name,
                type=e.type,
                context=e.context,
                chunk_uid=ec.chunk.uid,
                source=SourceRef(doc_title=ec.chunk.doc_title, chapter=ec.chunk.chapter, section=ec.chunk.section),
            )
        )

print(f"{len(tier1_entities)} Tier-1 entity mentions")

unique_names = sorted({entity_key(e.name): e.name for e in tier1_entities}.items())
print(f"{len(unique_names)} unique entity names across the corpus")

In [ ]:
class DictionaryDefinition(BaseModel):
    canonical_name: str
    type: SemanticType
    definition: str = Field(min_length=1, max_length=400)


_DEFINITION_SYSTEM_PROMPT = f"""You are a medical dictionary. Given a medical term, produce its \
canonical name, semantic type, and a concise (1-3 sentence) definition, as would appear in a \
standard medical reference. `type` must be exactly one of: {", ".join(SEMANTIC_TYPES)}.

Respond with ONLY a JSON object: {{"canonical_name": ..., "type": ..., "definition": ...}}
No prose, no markdown fences, no explanation — JSON only.

NOTE: you are a general-purpose LLM standing in for a licensed UMLS/SNOMED lookup — this \
definition is illustrative, not a verified controlled-vocabulary entry."""


def define_term(name: str, max_retries: int = 2) -> DictionaryDefinition:
    last_err = None
    for _ in range(max_retries + 1):
        raw = llm.call_json(_DEFINITION_SYSTEM_PROMPT, name, max_new_tokens=512)
        try:
            return DictionaryDefinition.model_validate_json(raw)
        except ValidationError as e:
            last_err = e
    raise last_err


tier3_definitions: dict[str, DictionaryDefinition] = {}
def_failures = 0
for key, name in tqdm(unique_names, desc="defining terms"):
    try:
        tier3_definitions[key] = define_term(name)
    except Exception as e:
        def_failures += 1
        tqdm.write(f"skipping definition for {name!r}: {e}")

print(f"defined {len(tier3_definitions)}/{len(unique_names)} unique term(s), {def_failures} failure(s)")

## Step 4 — Relationship extraction

Per chunk, one local Qwen3.5-8B call over the chunk text *and* its already-extracted
entity list (grounds relation endpoints to known entities instead of
hallucinating new ones — same closed-list philosophy as `domain_kg/guards.py`
and `mcq_generation.ipynb`'s fixed `RELATION_TAXONOMY`) produces a directed
relation graph per chunk (the paper's "Meta-MedGraph").

In [ ]:
RELATION_TAXONOMY = (
    "PRODUCES",
    "REGULATES",
    "INHIBITS",
    "STIMULATES",
    "ACTS_ON",
    "PART_OF",
    "LOCATED_IN",
    "INNERVATES",
    "SUPPLIES",
    "CAUSES",
    "DIAGNOSED_BY",
    "TREATED_BY",
    "PREREQUISITE_OF",
    "TRANSDUCES",
    "MODULATES",
    "CONNECTS_TO",
)
RelationType = Literal[
    "PRODUCES", "REGULATES", "INHIBITS", "STIMULATES", "ACTS_ON", "PART_OF",
    "LOCATED_IN", "INNERVATES", "SUPPLIES", "CAUSES", "DIAGNOSED_BY",
    "TREATED_BY", "PREREQUISITE_OF", "TRANSDUCES", "MODULATES", "CONNECTS_TO",
]


class RelationTriple(BaseModel):
    source: str = Field(min_length=1, description="must match an entity name from the provided list")
    relation: RelationType
    target: str = Field(min_length=1, description="must match an entity name from the provided list")
    description: str = Field(min_length=1, max_length=250)


class RelationExtractionResult(BaseModel):
    relations: list[RelationTriple]


_RELATION_SYSTEM_PROMPT = f"""You are a medical knowledge graph relation extractor. Given a \
passage and a list of entities already extracted from it, identify directed relationships \
between those entities.

`relation` must use exactly one relation from this fixed taxonomy — never invent new relations, \
never use a passive/inverse form (e.g. "STIMULATED_BY"), always express relations in the \
subject-acts-on-object direction: {", ".join(RELATION_TAXONOMY)}.

`source` and `target` MUST each exactly match one of the provided entity names (case-insensitive \
match is fine, but do not introduce entities not in the list). Skip a relationship if neither \
endpoint is in the list.

Respond with ONLY a JSON object: {{"relations": [{{"source": ..., "relation": ..., "target": ..., \
"description": ...}}, ...]}}
No prose, no markdown fences, no explanation — JSON only."""


def extract_relations(chunk_text: str, entity_names: list[str], max_retries: int = 2) -> RelationExtractionResult:
    user_msg = f"ENTITIES: {json.dumps(entity_names)}\n\nPASSAGE:\n{chunk_text}"
    last_err = None
    for _ in range(max_retries + 1):
        raw = llm.call_json(_RELATION_SYSTEM_PROMPT, user_msg, max_new_tokens=2048)
        try:
            return RelationExtractionResult.model_validate_json(raw)
        except ValidationError as e:
            last_err = e
    raise last_err

In [ ]:
@dataclass
class ChunkGraph:
    chunk: SourceChunk
    entities: list[Entity]
    relations: list[RelationTriple]


chunk_graphs: list[ChunkGraph] = []
rel_failures = 0
for ec in tqdm(extracted_chunks, desc="extracting relations"):
    if not ec.entities:
        chunk_graphs.append(ChunkGraph(chunk=ec.chunk, entities=[], relations=[]))
        continue
    names = [e.name for e in ec.entities]
    try:
        result = extract_relations(ec.chunk.text, names)
    except Exception as e:
        rel_failures += 1
        tqdm.write(f"skipping relations for {ec.chunk.uid}: {e}")
        chunk_graphs.append(ChunkGraph(chunk=ec.chunk, entities=ec.entities, relations=[]))
        continue
    # ground: keep only relations whose endpoints match an extracted entity name
    name_keys = {entity_key(n) for n in names}
    grounded = [r for r in result.relations if entity_key(r.source) in name_keys and entity_key(r.target) in name_keys]
    chunk_graphs.append(ChunkGraph(chunk=ec.chunk, entities=ec.entities, relations=grounded))

total_relations = sum(len(cg.relations) for cg in chunk_graphs)
print(f"extracted relations for {len(chunk_graphs) - rel_failures}/{len(extracted_chunks)} chunk(s), {rel_failures} failure(s)")
print(f"total grounded relations: {total_relations}")

In [ ]:
# Sanity check — spot-check a few relations against source text
for cg in chunk_graphs[:2]:
    print(cg.chunk.uid)
    for r in cg.relations[:5]:
        print(f"  {r.source} --[{r.relation}]--> {r.target}  ({r.description})")
    print()

## Step 5 — Graph assembly (tri-tier)

Assembled as a `networkx.MultiDiGraph`: node types `{entity, source,
definition}`, edge types `{<relation types from Step 4>, the_reference_of,
the_definition_of}`. Tier-1 entity nodes are deduped by normalized name (one
node per unique entity, all its chunk mentions/contexts kept as a list
attribute) so relations from different chunks land on the same node.

In [ ]:
import networkx as nx

G = nx.MultiDiGraph()


def tier1_node_id(name: str) -> str:
    return f"entity::{entity_key(name)}"


def source_node_id(source: SourceRef) -> str:
    return f"source::{entity_key(source.chapter)}::{entity_key(source.section)}"


def definition_node_id(name: str) -> str:
    return f"definition::{entity_key(name)}"


# Tier-1 entity nodes + Tier-2 source nodes + the_reference_of edges
for cg in chunk_graphs:
    src_ref = SourceRef(doc_title=cg.chunk.doc_title, chapter=cg.chunk.chapter, section=cg.chunk.section)
    sid = source_node_id(src_ref)
    if sid not in G:
        G.add_node(sid, tier=2, kind="source", doc_title=src_ref.doc_title, chapter=src_ref.chapter, section=src_ref.section)

    for e in cg.entities:
        eid = tier1_node_id(e.name)
        if eid not in G:
            G.add_node(eid, tier=1, kind="entity", name=e.name, type=e.type, contexts=[])
        G.nodes[eid]["contexts"].append({"chunk_uid": cg.chunk.uid, "context": e.context})
        G.add_edge(eid, sid, key="the_reference_of", relation="the_reference_of")

# Tier-3 definition nodes + the_definition_of edges
for key, defn in tier3_definitions.items():
    eid = f"entity::{key}"
    if eid not in G:
        continue  # definition for a name that ended up with zero surviving entity nodes
    did = definition_node_id(defn.canonical_name)
    G.add_node(did, tier=3, kind="definition", canonical_name=defn.canonical_name, type=defn.type, definition=defn.definition)
    G.add_edge(eid, did, key="the_definition_of", relation="the_definition_of")

# Relations between Tier-1 entities
for cg in chunk_graphs:
    for r in cg.relations:
        sid, tid = tier1_node_id(r.source), tier1_node_id(r.target)
        if sid not in G or tid not in G:
            continue
        G.add_edge(sid, tid, key=f"{r.relation}::{cg.chunk.uid}", relation=r.relation, description=r.description, chunk_uid=cg.chunk.uid)

print(f"graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
tier_counts = {}
for _, data in G.nodes(data=True):
    tier_counts[data["tier"]] = tier_counts.get(data["tier"], 0) + 1
print(f"nodes per tier: {tier_counts}")

## Step 6 — Tagging + hierarchical clustering

Per chunk-graph, one local Qwen3.5-8B call summarizes it against a fixed medical tag
set (paper 2.2). Tag summaries are then embedded (`sentence-transformers`,
same `all-MiniLM-L6-v2` model as `src/ingestion/semantic_chunk.py`, since the
local LLM here isn't an embedding model) and agglomeratively merged —
top 20% most similar pairs per iteration, same as the paper — into a small
multi-layer tag tree. Capped at 3 layers here (paper allows up to 12; this
corpus is 32 chapters, not thousands of documents).

In [ ]:
TAG_CATEGORIES = (
    "Anatomy",
    "Physiology",
    "Pathology",
    "Clinical Presentation",
    "Pharmacology",
    "Biochemistry",
    "Metabolism",
    "Molecular Genetics",
)


class TagSummary(BaseModel):
    tags: list[str] = Field(min_length=1, max_length=4, description="subset of the given categories that apply")
    summary: str = Field(min_length=1, max_length=300)


_TAG_SYSTEM_PROMPT = f"""Generate a structured summary from the provided medical content, \
strictly adhering to these categories: {", ".join(TAG_CATEGORIES)}.

Pick 1-4 categories that best describe the content's tags, and write a short (1-3 sentence) \
summary of what this section covers.

Respond with ONLY a JSON object: {{"tags": [...], "summary": ...}}
No prose, no markdown fences, no explanation — JSON only."""


def tag_chunk(chunk_text: str, section: str, max_retries: int = 2) -> TagSummary:
    user_msg = f"SECTION: {section}\n\n{chunk_text}"
    last_err = None
    for _ in range(max_retries + 1):
        raw = llm.call_json(_TAG_SYSTEM_PROMPT, user_msg, max_new_tokens=512)
        try:
            return TagSummary.model_validate_json(raw)
        except ValidationError as e:
            last_err = e
    raise last_err


@dataclass
class TaggedChunk:
    chunk_uid: str
    tags: list[str]
    summary: str


tagged_chunks: list[TaggedChunk] = []
tag_failures = 0
for cg in tqdm(chunk_graphs, desc="tagging chunks"):
    try:
        ts = tag_chunk(cg.chunk.text, cg.chunk.section)
    except Exception as e:
        tag_failures += 1
        tqdm.write(f"skipping tags for {cg.chunk.uid}: {e}")
        continue
    tagged_chunks.append(TaggedChunk(chunk_uid=cg.chunk.uid, tags=ts.tags, summary=ts.summary))

print(f"tagged {len(tagged_chunks)}/{len(chunk_graphs)} chunk(s), {tag_failures} failure(s)")

In [ ]:
import gc

import numpy as np
import torch
from sentence_transformers import SentenceTransformer


def get_device() -> str:
    if torch.cuda.is_available():
        return "cuda"
    if torch.backends.mps.is_available():
        return "mps"
    return "cpu"


embed_model = SentenceTransformer("all-MiniLM-L6-v2", device=get_device())


def embed_texts(texts: list[str]) -> np.ndarray:
    return embed_model.encode(texts, convert_to_numpy=True, normalize_embeddings=True)


@dataclass
class TagCluster:
    layer: int
    cluster_id: str
    summary: str
    tags: list[str]
    members: list[str]  # chunk_uids (layer 0) or child cluster_ids (layer > 0)


def cluster_layer(clusters: list[TagCluster], layer: int, top_frac: float = 0.2) -> list[TagCluster]:
    if len(clusters) <= 1:
        return clusters
    texts = [" ".join(c.tags) + " " + c.summary for c in clusters]
    embs = embed_texts(texts)
    sims = embs @ embs.T
    n = len(clusters)
    pairs = [(sims[i, j], i, j) for i in range(n) for j in range(i + 1, n)]
    pairs.sort(reverse=True)
    top_k = max(1, int(len(pairs) * top_frac))
    threshold = pairs[top_k - 1][0] if pairs else 1.0

    parent: dict[int, int] = {i: i for i in range(n)}

    def find(x: int) -> int:
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    for sim, i, j in pairs:
        if sim >= threshold:
            ri, rj = find(i), find(j)
            if ri != rj:
                parent[ri] = rj

    groups: dict[int, list[int]] = {}
    for i in range(n):
        groups.setdefault(find(i), []).append(i)

    merged: list[TagCluster] = []
    for gi, (root, members) in enumerate(groups.items()):
        if len(members) == 1:
            c = clusters[members[0]]
            merged.append(TagCluster(layer=layer, cluster_id=f"L{layer}::{gi:03d}", summary=c.summary, tags=c.tags, members=[c.cluster_id]))
            continue
        member_clusters = [clusters[m] for m in members]
        merged_tags = sorted({t for c in member_clusters for t in c.tags})
        combined_summary = " | ".join(c.summary for c in member_clusters)
        merged.append(
            TagCluster(
                layer=layer,
                cluster_id=f"L{layer}::{gi:03d}",
                summary=combined_summary[:500],
                tags=merged_tags,
                members=[c.cluster_id for c in member_clusters],
            )
        )
    return merged


MAX_LAYERS = 3
layer0 = [TagCluster(layer=0, cluster_id=tc.chunk_uid, summary=tc.summary, tags=tc.tags, members=[tc.chunk_uid]) for tc in tagged_chunks]
tag_layers: list[list[TagCluster]] = [layer0]
for layer in range(1, MAX_LAYERS + 1):
    prev = tag_layers[-1]
    if len(prev) <= 1:
        break
    nxt = cluster_layer(prev, layer)
    tag_layers.append(nxt)
    if len(nxt) == len(prev):
        break  # converged, no further merging happening

embed_model = None
gc.collect()
if torch.backends.mps.is_available():
    torch.mps.empty_cache()

for i, layer in enumerate(tag_layers):
    print(f"layer {i}: {len(layer)} cluster(s)")

## Step 7 — U-Retrieval demo

Top-down: match the query's own tag-summary against the top tag layer,
descend to the most similar cluster at each layer down to the chunk-level
leaves, gather those chunks' entities + relations as the subgraph, answer
grounded in it. Bottom-up: one refinement pass folding back the parent
layer's broader summary, per the paper's U-Retrieval (2.3).

In [ ]:
embed_model = SentenceTransformer("all-MiniLM-L6-v2", device=get_device())


def query_tag_summary(question: str) -> TagSummary:
    return tag_chunk(question, section="query")


def top_down_retrieve(question: str) -> tuple[list[str], list[TagCluster]]:
    """Returns (leaf chunk_uids, path of clusters visited from top layer down)."""
    q_tag = query_tag_summary(question)
    q_text = " ".join(q_tag.tags) + " " + q_tag.summary
    q_emb = embed_texts([q_text])[0]

    path: list[TagCluster] = []
    current_layer = tag_layers[-1]
    while True:
        texts = [" ".join(c.tags) + " " + c.summary for c in current_layer]
        embs = embed_texts(texts)
        sims = embs @ q_emb
        best_idx = int(np.argmax(sims))
        best = current_layer[best_idx]
        path.append(best)
        if best.layer == 0:
            return best.members, path
        # descend: gather this cluster's member cluster_ids at the layer below
        below = tag_layers[best.layer - 1]
        current_layer = [c for c in below if c.cluster_id in best.members]
        if not current_layer:
            return [], path


def gather_subgraph(chunk_uids: list[str]) -> tuple[list[str], list[tuple[str, str, str, str]]]:
    chunk_uid_set = set(chunk_uids)
    relevant_cgs = [cg for cg in chunk_graphs if cg.chunk.uid in chunk_uid_set]
    entity_names = sorted({e.name for cg in relevant_cgs for e in cg.entities})
    triples = [(r.source, r.relation, r.target, r.description) for cg in relevant_cgs for r in cg.relations]
    return entity_names, triples


_ANSWER_SYSTEM_PROMPT = """You are a medical Q&A assistant. Answer the question using ONLY the \
provided graph of entities and relationships extracted from Harper's Illustrated Biochemistry. \
If the graph doesn't contain enough information, say so explicitly rather than guessing. Cite \
the specific relations you used."""


def answer_with_graph(question: str, entity_names: list[str], triples: list[tuple[str, str, str, str]]) -> str:
    graph_text = "\n".join(f"{s} --[{r}]--> {t} ({d})" for s, r, t, d in triples)
    user_msg = f"QUESTION: {question}\n\nGRAPH:\n{graph_text}\n\nAnswer the question using the graph above."
    return llm.call_text(_ANSWER_SYSTEM_PROMPT, user_msg, max_new_tokens=1024)


def refine_with_parent_summary(question: str, prior_answer: str, parent_summary: str) -> str:
    user_msg = (
        f"QUESTION: {question}\nLAST RESPONSE: {prior_answer}\nSUMMARY: {parent_summary}\n\n"
        "Adjust the response using the broader context in SUMMARY if it adds relevant information; "
        "otherwise keep the response as-is."
    )
    return llm.call_text(_ANSWER_SYSTEM_PROMPT, user_msg, max_new_tokens=1024)


def u_retrieval(question: str) -> str:
    leaf_chunk_uids, path = top_down_retrieve(question)
    entity_names, triples = gather_subgraph(leaf_chunk_uids)
    answer = answer_with_graph(question, entity_names, triples)
    # bottom-up: fold in one parent-layer summary if available
    if len(path) >= 2:
        parent = path[-2]
        answer = refine_with_parent_summary(question, answer, parent.summary)
    return answer

In [ ]:
DEMO_QUESTIONS = [
    "What is the role of ATP in bioenergetics?",
    "How does the citric acid cycle connect carbohydrate, lipid, and amino acid metabolism?",
    "What is the biochemical basis of glycogen storage diseases?",
]

for q in DEMO_QUESTIONS:
    print(f"Q: {q}")
    print(u_retrieval(q))
    print("\n" + "-" * 80 + "\n")

## Step 8 — Export

Graph exported as JSON (nodes/edges, easy to inspect/reload) and GraphML
(openable in Gephi/Cytoscape). A markdown summary of node/edge counts per
tier and per chapter is written alongside.

In [ ]:
from networkx.readwrite import json_graph

graph_json_path = OUTPUT_DIR / "graph.json"
graph_json_path.write_text(json.dumps(json_graph.node_link_data(G, edges="edges"), indent=2), encoding="utf-8")

# GraphML requires scalar/string attribute values — flatten list/dict attrs first
G_export = G.copy()
for _, data in G_export.nodes(data=True):
    if "contexts" in data:
        data["contexts"] = json.dumps(data["contexts"])
graphml_path = OUTPUT_DIR / "graph.graphml"
nx.write_graphml(G_export, graphml_path)

chapter_counts: dict[str, int] = {}
for cg in chunk_graphs:
    chapter_counts[cg.chunk.chapter] = chapter_counts.get(cg.chunk.chapter, 0) + len(cg.entities)

summary_lines = [
    "# MedGraphRAG KG — build summary",
    "",
    f"- Total nodes: {G.number_of_nodes()}",
    f"- Total edges: {G.number_of_edges()}",
    f"- Nodes per tier: {tier_counts}",
    f"- Tag layers: {[len(l) for l in tag_layers]}",
    "",
    "## Entity mentions per chapter",
    "",
]
for chapter, count in chapter_counts.items():
    summary_lines.append(f"- {chapter}: {count} entity mentions")

summary_path = OUTPUT_DIR / "summary.md"
summary_path.write_text("\n".join(summary_lines), encoding="utf-8")

print(f"wrote {graph_json_path}")
print(f"wrote {graphml_path}")
print(f"wrote {summary_path}")

In [ ]:
# Round-trip check: reload GraphML, confirm counts match
G_reloaded = nx.read_graphml(graphml_path)
assert G_reloaded.number_of_nodes() == G.number_of_nodes(), (G_reloaded.number_of_nodes(), G.number_of_nodes())
assert G_reloaded.number_of_edges() == G.number_of_edges(), (G_reloaded.number_of_edges(), G.number_of_edges())
print("round-trip OK:", G_reloaded.number_of_nodes(), "nodes,", G_reloaded.number_of_edges(), "edges")

In [ ]:
llm.unload()